# Eyecite vs Haiku Citation Extraction Comparison

Per-opinion comparison of eyecite 2.7.6 vs Haiku 4.5 stage 1 extraction on the 0518 benchmark (383 SCOTUS + circuit-court opinions).

- **Haiku output**: existing `experiments_05182026/data/output/parsed_results.csv` (re-used)
- **Eyecite output**: generated by `run_eyecite.py` in this folder

Focus per user direction: start with opinions where Haiku extracted zero citations.

In [1]:
import json
import os
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

DATA = 'data'
OPINIONS = f'{DATA}/opinion_texts'

benchmark = pd.read_csv(f'{DATA}/benchmark_cluster_ids.csv')['cluster_id'].astype(str).tolist()
meta = pd.read_csv(f'{DATA}/citing_metadata.csv', dtype={'cluster_id': str})
haiku = pd.read_csv(f'{DATA}/haiku_extractions.csv', dtype={'citing_cluster_id': str})
eyecite = pd.read_csv(f'{DATA}/eyecite_extractions.csv', dtype={'citing_cluster_id': str})

print(f'benchmark: {len(benchmark)} opinions')
print(f'haiku:     {len(haiku):>6} citations across {haiku.citing_cluster_id.nunique()} opinions')
print(f'eyecite:   {len(eyecite):>6} citations across {eyecite.citing_cluster_id.nunique()} opinions')

benchmark: 383 opinions
haiku:      13044 citations across 378 opinions
eyecite:    15604 citations across 382 opinions


## Top-line stats

In [2]:
h_per = haiku.groupby('citing_cluster_id').size()
e_per = eyecite.groupby('citing_cluster_id').size()

stats = pd.DataFrame({
    'haiku':   [len(haiku), haiku.citing_cluster_id.nunique(), h_per.median(), h_per.mean(), h_per.max(),
                haiku.mainCitationString.notna().mean()*100, haiku.caseName.notna().mean()*100],
    'eyecite': [len(eyecite), eyecite.citing_cluster_id.nunique(), e_per.median(), e_per.mean(), e_per.max(),
                eyecite.mainCitationString.notna().mean()*100, eyecite.caseName.notna().mean()*100],
}, index=['total_citations', 'opinions_covered', 'per_opinion_median',
         'per_opinion_mean', 'per_opinion_max', 'mainCitation_nonnull_%', 'caseName_nonnull_%']).round(2)
stats

,haiku,eyecite
total_citations,13044.00,15604.00
opinions_covered,378.00,382.00
per_opinion_median,25.00,28.00
per_opinion_mean,34.51,40.85
per_opinion_max,169.00,355.00
mainCitation_nonnull_%,99.30,98.31
caseName_nonnull_%,91.18,88.30


## Coverage gap — which opinions did each approach miss?

In [3]:
b = set(benchmark)
h_ids = set(haiku.citing_cluster_id)
e_ids = set(eyecite.citing_cluster_id)

missing_haiku   = sorted(b - h_ids)
missing_eyecite = sorted(b - e_ids)

print(f'Missing from Haiku   ({len(missing_haiku)}): {missing_haiku}')
print(f'Missing from eyecite ({len(missing_eyecite)}): {missing_eyecite}')
print(f'Missing from BOTH    : {sorted((b - h_ids) & (b - e_ids))}')

Missing from Haiku   (5): ['106447', '106548', '110077', '111094', '121162']
Missing from eyecite (1): ['4450553']
Missing from BOTH    : []


## Focus 1 — Opinions where Haiku extracted zero citations

Show metadata and opinion size for each, plus how many citations eyecite found instead.

In [4]:
def opinion_size(cid):
    path = f'{OPINIONS}/{cid}.txt'
    return os.path.getsize(path) if os.path.exists(path) else 0

haiku_zero = pd.DataFrame({'cluster_id': missing_haiku})
haiku_zero = haiku_zero.merge(meta[['cluster_id', 'court', 'case_name', 'year', 'num_authorities', 'opinion_text_length']],
                              on='cluster_id', how='left')
haiku_zero['file_size_kb'] = haiku_zero['cluster_id'].apply(lambda c: opinion_size(c) // 1024)
haiku_zero['eyecite_found'] = haiku_zero['cluster_id'].map(e_per).fillna(0).astype(int)
haiku_zero.sort_values('file_size_kb', ascending=False)

,cluster_id,court,case_name,year,num_authorities,opinion_text_length,file_size_kb,eyecite_found
3,111094,scotus,Pennhurst State School and Hospital v. Halderman,1984,151,380415,373,161
2,110077,scotus,Cannon v. University of Chicago,1979,147,355505,348,172
1,106548,scotus,Fay v. Noia,1963,196,353112,345,218
0,106447,scotus,Glidden Co. v. Zdanok,1962,172,308551,302,184
4,121162,scotus,Utah v. Evans,2002,28,243259,239,29


**Observation.** All five Haiku-zero opinions are very long SCOTUS cases (>240 KB each, biggest ~382 KB). Likely Haiku batch failures from token-limit or pagination issues, not an extraction-quality problem. Eyecite handled them fine in ~ms each.

Below: sample of citations eyecite found in those opinions Haiku produced nothing for.

In [5]:
for cid in missing_haiku:
    subset = eyecite[eyecite.citing_cluster_id == cid]
    case_name = meta.loc[meta.cluster_id == cid, 'case_name'].values
    case_name = case_name[0] if len(case_name) else '?'
    print(f'\n--- {cid}: {case_name} ({len(subset)} eyecite extractions) ---')
    print(subset[['mainCitationString', 'caseName', 'section_ids']].head(5).to_string(index=False))


--- 106447: Glidden Co. v. Zdanok (184 eyecite extractions) ---
mainCitationString                  caseName                                                                                                 section_ids
      370 U.S. 530                       NaN                                                                                                      ["S1"]
      368 U.S. 973                     Court                                                                                                      ["S1"]
      279 U.S. 438            Bakelite Corp. ["S1", "S4", "S5", "S6", "S7", "S11", "S12", "S13", "S16", "S18", "S22", "S24", "S30", "S31", "S33", "S37"]
      289 U.S. 553 Williams v. United States                                                ["S1", "S6", "S8", "S12", "S13", "S22", "S30", "S33", "S37"]
      366 U.S. 712                      Term                                                                                               ["S1", "S33"]

--- 106548: Fay 

## Focus 2 — The single opinion eyecite missed (4450553)

Check whether the opinion text has unusual encoding (NBSP, smart quotes, etc.) that defeats eyecite's regex.

In [6]:
cid = '4450553'
with open(f'{OPINIONS}/{cid}.txt') as f:
    text = f.read()

print(f'cluster_id: {cid}')
print(f'size: {len(text)} chars')
print(f'  regular spaces:        {text.count(chr(0x20))}')
print(f'  non-breaking spaces:   {text.count(chr(0xA0))}')
print(f'  newlines:              {text.count(chr(0x0A))}')
print()
print('First 500 chars (repr to show invisible chars):')
print(repr(text[:500]))
print()
haiku_for_4450553 = haiku[haiku.citing_cluster_id == cid][['mainCitationString', 'caseName', 'section_ids']]
print(f'Haiku extracted {len(haiku_for_4450553)} citations from this opinion:')
print(haiku_for_4450553.head(5).to_string(index=False))

cluster_id: 4450553
size: 49717 chars
  regular spaces:        3987
  non-breaking spaces:   7550
  newlines:              1188

First 500 chars (repr to show invisible chars):
'16‐1111‐cr\xa0\nUnited\xa0States\xa0v.\xa0Latchman\xa0Singh\xa0\n                                                      \xa0\n                                   UNITED\xa0STATES\xa0COURT\xa0OF\xa0APPEALS\xa0\n                                       FOR\xa0THE\xa0SECOND\xa0CIRCUIT\xa0\n\n                                            August\xa0Term\xa02016\xa0\n\n             (Argued:\xa0February\xa023,\xa02017\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 Decided:\xa0\xa0\xa0December\xa012,\xa02017)\xa0\n                                   Docket\xa0No.\xa016‐1111‐cr\xa0\n\n                                    \xa0       \xa0     \xa0       '

Haiku extracted 35 citations from this opinion:
mainCitationString                   caseName                  section_ids
       552 U.S. 38      Gall v. 

**Observation.** The opinion uses non-breaking spaces (U+00A0) almost everywhere a regular space would appear. Eyecite's reporter regexes match `\s` for inter-token whitespace, so this *should* work in principle — but the high NBSP density likely interacts with tokenizer assumptions. Worth a targeted reproduction; this is a real-world eyecite gap on a single opinion, useful as an example.

## Per-opinion overlap (for opinions in both outputs)

For each opinion in both Haiku and eyecite output, compute set-overlap on `mainCitationString`. Note the comparison is on *exact citation string* — caseName differences are not penalized here.

In [7]:
shared = h_ids & e_ids

h_sets = {cid: set(haiku.loc[haiku.citing_cluster_id == cid, 'mainCitationString'].dropna())
          for cid in shared}
e_sets = {cid: set(eyecite.loc[eyecite.citing_cluster_id == cid, 'mainCitationString'].dropna())
          for cid in shared}

rows = []
for cid in shared:
    h, e = h_sets[cid], e_sets[cid]
    rows.append({
        'cluster_id': cid,
        'haiku_count': len(h),
        'eyecite_count': len(e),
        'both': len(h & e),
        'haiku_only': len(h - e),
        'eyecite_only': len(e - h),
        'jaccard': len(h & e) / max(1, len(h | e)),
    })
overlap = pd.DataFrame(rows)
print('Overlap distribution across', len(overlap), 'opinions:')
overlap[['haiku_count', 'eyecite_count', 'both', 'haiku_only', 'eyecite_only', 'jaccard']].describe().round(2)

Overlap distribution across 377 opinions:


,haiku_count,eyecite_count,both,haiku_only,eyecite_only,jaccard
count,377.00,377.00,377.00,377.00,377.00,377.00
mean,34.27,38.67,30.98,3.28,7.68,0.78
std,29.15,35.57,27.95,7.07,17.91,0.23
min,1.00,1.00,1.00,0.00,0.00,0.02
25%,15.00,16.00,12.00,0.00,0.00,0.64
50%,24.00,27.00,22.00,1.00,3.00,0.85
75%,46.00,50.00,41.00,4.00,7.00,0.98
max,167.00,355.00,155.00,81.00,243.00,1.00


In [8]:
print('Jaccard bucket distribution:')
print(pd.cut(overlap.jaccard, bins=[0, 0.25, 0.5, 0.75, 0.9, 1.0], include_lowest=True).value_counts().sort_index())
print()
print('Top-10 lowest-agreement opinions (Jaccard):')
lowest = overlap.merge(meta[['cluster_id', 'court', 'case_name', 'year']], left_on='cluster_id', right_on='cluster_id').sort_values('jaccard').head(5)
lowest[['cluster_id', 'court', 'year', 'case_name', 'haiku_count', 'eyecite_count', 'both', 'haiku_only', 'eyecite_only', 'jaccard']]

Jaccard bucket distribution:
jaccard
(-0.001, 0.25]     13
(0.25, 0.5]        43
(0.5, 0.75]        92
(0.75, 0.9]        73
(0.9, 1.0]        156
Name: count, dtype: int64

Top-10 lowest-agreement opinions (Jaccard):


,cluster_id,court,year,case_name,haiku_count,eyecite_count,both,haiku_only,eyecite_only,jaccard
358,2233552,michctapp,1984,Bishop v. St John Hospital,24,25,1,23,24,0.020833
32,10610182,lactapp,2019,State of Louisiana v. Teddy R. Magee,21,18,1,20,17,0.026316
147,9382158,fladistctapp,2023,"TOTAL QUALITY LOGISTICS, LLC v. TRADE LINK CAPITAL, INC.",11,12,1,10,11,0.045455
131,1801425,michctapp,1978,"Bushman v. Burns Clinic Medical Center, P. C.",26,28,4,22,24,0.080000
289,10415317,cand,2025,"GMC Semitech Co., Ltd. v. Capital Asset Exchange and Trading, LLC",17,23,5,12,18,0.142857


## Spot-check disagreements

Pick a few low-agreement opinions and look at the eyecite-only and haiku-only sets to characterize the disagreements.

In [9]:
def show_disagreements(cid, n=5):
    h, e = h_sets[cid], e_sets[cid]
    case_name = meta.loc[meta.cluster_id == cid, 'case_name'].values
    case_name = case_name[0] if len(case_name) else '?'
    print(f'\n=== {cid}: {case_name} ===')
    print(f'haiku={len(h)}, eyecite={len(e)}, both={len(h & e)}, jaccard={len(h & e)/max(1,len(h|e)):.2f}')
    print(f'\nEyecite-only (top {n}):')
    for c in sorted(e - h)[:n]:
        rows = eyecite[(eyecite.citing_cluster_id == cid) & (eyecite.mainCitationString == c)]
        if len(rows):
            print(f'  {c:35s} | caseName: {rows.iloc[0].caseName!r}')
    print(f'\nHaiku-only (top {n}):')
    for c in sorted(h - e)[:n]:
        rows = haiku[(haiku.citing_cluster_id == cid) & (haiku.mainCitationString == c)]
        if len(rows):
            print(f'  {c:35s} | caseName: {rows.iloc[0].caseName!r}')

for cid in lowest['cluster_id'].head(3):
    show_disagreements(cid)


=== 2233552: Bishop v. St John Hospital ===
haiku=24, eyecite=25, both=1, jaccard=0.02

Eyecite-only (top 5):
  114 MichApp 216                     | caseName: 'Carbonell v. Bluhm'
  121 MichApp 615                     | caseName: 'Roe, Inc'
  125 MichApp 724                     | caseName: 'Cook v. Detroit'
  138 NW2d 503                        | caseName: nan
  140 Mich.App. 720                   | caseName: nan

Haiku-only (top 5):
  114 Mich App 216                    | caseName: 'Carbonell v Bluhm'
  121 Mich App 615                    | caseName: 'Joba Construction Co, Inc v Burns & Roe, Inc'
  125 Mich App 724                    | caseName: 'Cook v Detroit'
  138 N.W.2d 503                      | caseName: 'Fogel v Sinai Hospital of Detroit'
  140 Mich. App. 720                  | caseName: 'Bishop v. St John Hospital'

=== 10610182: State of Louisiana v. Teddy R. Magee ===
haiku=21, eyecite=18, both=1, jaccard=0.03

Eyecite-only (top 5):
  12 So.3d 386                        |

## Side-by-side comparison for a sample opinion

The per-opinion overlap section above shows aggregate Jaccard; the spot-check section shows haiku-only and eyecite-only citations. This section shows the full picture for a single opinion — both agreement and disagreement aligned by citation.

In [10]:
def side_by_side(cid):
    """Return a joined table of every citation either extractor found for this opinion,
    with both extractors' caseName + section_ids shown side-by-side."""
    h = haiku[haiku.citing_cluster_id == cid][['mainCitationString', 'caseName', 'section_ids']].rename(
        columns={'caseName': 'haiku_caseName', 'section_ids': 'haiku_sections'}
    )
    e = eyecite[eyecite.citing_cluster_id == cid][['mainCitationString', 'caseName', 'section_ids']].rename(
        columns={'caseName': 'eyecite_caseName', 'section_ids': 'eyecite_sections'}
    )
    merged = h.merge(e, on='mainCitationString', how='outer', indicator=True)
    merged['status'] = merged['_merge'].map({'both': 'both', 'left_only': 'haiku_only', 'right_only': 'eyecite_only'})
    merged = merged.drop(columns='_merge')
    return merged.sort_values(['status', 'mainCitationString'])

# Pick a high-agreement opinion (Jaccard ~1.0) and a low-agreement one
high_cid = overlap.sort_values('jaccard', ascending=False).iloc[0]['cluster_id']
low_cid  = overlap.sort_values('jaccard').iloc[0]['cluster_id']

case_high = meta.loc[meta.cluster_id == high_cid, 'case_name'].values
case_high = case_high[0] if len(case_high) else '?'
case_low  = meta.loc[meta.cluster_id == low_cid, 'case_name'].values
case_low  = case_low[0] if len(case_low) else '?'

print(f'High-agreement example — {high_cid}: {case_high}')
print(f'  jaccard = {overlap.loc[overlap.cluster_id == high_cid, "jaccard"].iloc[0]:.2f}')
side_by_side(high_cid).head(5)

High-agreement example — 7844916: State v. Houston
  jaccard = 1.00


,mainCitationString,haiku_caseName,haiku_sections,eyecite_caseName,eyecite_sections,status
0,384 So.2d 355,State v. Bonanno,"[""S4""]",State v. Bonanno,"[""S4""]",both
1,398 So.2d 1049,State v. Jones,"[""S4""]",State v. Jones,"[""S4""]",both
2,419 So.2d 475,State v. Landos,"[""S4""]",State v. Landos,"[""S4""]",both
3,433 So.2d 688,State v. Smith,"[""S4""]",State v. Smith,"[""S4""]",both
4,475 So.2d 1094,Overmier v. Traylor,"[""S3""]",Overmier v. Traylor,"[""S3""]",both


In [11]:
print(f'Low-agreement example — {low_cid}: {case_low}')
print(f'  jaccard = {overlap.loc[overlap.cluster_id == low_cid, "jaccard"].iloc[0]:.2f}')
side_by_side(low_cid).head(5)

Low-agreement example — 2233552: Bishop v. St John Hospital
  jaccard = 0.02


,mainCitationString,haiku_caseName,haiku_sections,eyecite_caseName,eyecite_sections,status
0,114 Mich App 216,Carbonell v Bluhm,"[""S3""]",NaN,NaN,haiku_only
2,121 Mich App 615,"Joba Construction Co, Inc v Burns & Roe, Inc","[""S3""]",NaN,NaN,haiku_only
4,125 Mich App 724,Cook v Detroit,"[""S3""]",NaN,NaN,haiku_only
6,138 N.W.2d 503,Fogel v Sinai Hospital of Detroit,"[""S1"", ""S2"", ""S3""]",NaN,NaN,haiku_only
8,140 Mich. App. 720,Bishop v. St John Hospital,"[""S1""]",NaN,NaN,haiku_only


**Reading the side-by-side**: rows with `status='both'` are exact-string matches on `mainCitationString`. Differences in caseName (eyecite truncates multi-word plaintiffs / drops apostrophes; haiku doesn't) and section_ids (haiku undercounts occurrences; eyecite's resolver chains short/Id/supra back-references) are visible even on agreement rows. Rows with `status='haiku_only'` or `'eyecite_only'` are exclusive finds — often the same cite written differently (`Mich. App.` vs `MichApp`), or genuine recall gaps.

## Section ID assignment — do the two pipelines agree?

The same `split_into_sections()` function is called by both pipelines (Haiku via `run_two_stage.py`, eyecite via `run_eyecite.py`), so the section *boundaries* are identical. The difference is how each pipeline *assigns* citations to sections:

- **Haiku**: sees the annotated text (`[Section S1]\n...`) and is asked to report which section each citation appears in. LLM judgment.
- **Eyecite**: runs on raw text, gets char offsets; `offset_to_section_id()` maps each citation's `span()[0]` to whichever section's `[start_char, end_char)` range contains it.

Below: for citations both pipelines extracted, how often do the section_ids match?

In [12]:
def parse_sids(s):
    if pd.isna(s): return []
    try: return json.loads(s)
    except: return []

def aggregate(df):
    df = df.dropna(subset=['mainCitationString']).copy()
    df['_sids'] = df['section_ids'].apply(parse_sids)
    return df.groupby(['citing_cluster_id', 'mainCitationString'])['_sids'].apply(
        lambda lists: set().union(*[set(l) for l in lists])
    ).to_dict()

h_map = aggregate(haiku)
e_map = aggregate(eyecite)

shared_keys = set(h_map) & set(e_map)
exact = 0
mismatches = []
for k in shared_keys:
    if h_map[k] == e_map[k]:
        exact += 1
    else:
        mismatches.append((k, h_map[k], e_map[k]))

print(f'Shared (citing, citation) pairs in both:    {len(shared_keys)}')
print(f'  exact section_id match:                   {exact}  ({exact*100/len(shared_keys):.1f}%)')
print(f'  mismatch:                                 {len(mismatches)}  ({len(mismatches)*100/len(shared_keys):.1f}%)')

Shared (citing, citation) pairs in both:    11680
  exact section_id match:                   6860  (58.7%)
  mismatch:                                 4820  (41.3%)


In [13]:
def sec_num(s): return int(s[1:])

cats = {'haiku_superset': 0, 'eyecite_superset': 0, 'disjoint': 0, 'off_by_one_only': 0, 'partial_other': 0}
for k, h, e in mismatches:
    if e.issuperset(h):
        cats['eyecite_superset'] += 1
    elif h.issuperset(e):
        cats['haiku_superset'] += 1
    elif not (h & e):
        cats['disjoint'] += 1
    else:
        all_adj = (all(any(abs(sec_num(hh) - sec_num(ee)) <= 1 for ee in e) for hh in h) and
                   all(any(abs(sec_num(ee) - sec_num(hh)) <= 1 for hh in h) for ee in e))
        if all_adj:
            cats['off_by_one_only'] += 1
        else:
            cats['partial_other'] += 1

print('Mismatch breakdown:')
for k, v in cats.items():
    pct = v * 100 / len(mismatches)
    print(f'  {k:20s}  {v:>5}  ({pct:.1f}%)')

Mismatch breakdown:
  haiku_superset          352  (7.3%)
  eyecite_superset       3286  (68.2%)
  disjoint                589  (12.2%)
  off_by_one_only         129  (2.7%)
  partial_other           464  (9.6%)


**Interpretation.**

- **Eyecite-superset (68% of mismatches)** is the dominant pattern. Eyecite's resolver chains `Id.` / short / supra / reference back-citations to the canonical case, picking up every occurrence. Haiku's LLM is *under-counting* where a cited case appears across the opinion.
- **Pure off-by-one (3%)** is small — the section-offset issue you noticed is real but not the main story.
- **Haiku-superset (7%)** is the inverse: eyecite's resolver failed to chain a back-reference.
- **Disjoint (12%)** — investigated below.

In [14]:
# Verify disjoint cases by grep-locating the literal citation string in raw text.
sys_path_set = False
try:
    from utils.section_utils import split_into_sections
except ImportError:
    import sys
    sys.path.insert(0, '../citator-pipeline')
    from utils.section_utils import split_into_sections

import random
random.seed(0)

disjoint = [(k, h, e) for k, h, e in mismatches if not (h & e)]
print(f'Disjoint examples: {len(disjoint)}\n')

rows = []
for k, h, e in random.sample(disjoint, 10):
    cid, cite = k
    path = f'{OPINIONS}/{cid}.txt'
    if not os.path.exists(path): continue
    with open(path) as f: text = f.read()
    sections = split_into_sections(text)

    positions = []
    start = 0
    while True:
        idx = text.find(cite, start)
        if idx < 0: break
        positions.append(idx)
        start = idx + 1

    def which_sec(off):
        for s in sections:
            if s['start_char'] <= off < s['end_char']:
                return s['id']
        return None

    grep_sids = sorted({which_sec(p) for p in positions if which_sec(p)})
    rows.append({
        'cluster_id': cid,
        'citation': cite,
        'haiku': sorted(h),
        'eyecite': sorted(e),
        'literal_matches': len(positions),
        'grep_sections': grep_sids,
    })

pd.DataFrame(rows)

Disjoint examples: 589



,cluster_id,citation,haiku,eyecite,literal_matches,grep_sections
0,110484,601 F.2d 330,[S17],[S16],0,[]
1,2621076,191 U.S. 499,[S9],[S10],0,[]
2,7334436,534 F.3d 1017,[S3],[S1],1,[S1]
3,112786,494 U.S. 872,"[S45, S58]","[S46, S54]",0,[]
4,1749602,373 So.2d 488,[S2],"[S1, S3]",1,[S1]
5,4879558,478 F.3d 1092,[S3],[S4],0,[]
6,106729,369 U.S. 350,[S18],"[S17, S7]",0,[]
7,2369968,268 N.W.2d 683,[S11],[S12],1,[S12]
8,103633,212 U.S. 1,[S1],"[S13, S3]",2,"[S1, S3]"
9,2540705,283 U.S. 691,[S1],[S4],1,[S4]


**Interpretation of disjoint cases.** Where the literal citation string appears verbatim in the source text, the **grep-based section assignment agrees with eyecite, not Haiku**. Haiku's section_ids in these cases are LLM judgment errors — usually off by one near a section boundary, sometimes a section away.

Where the literal string has zero matches (e.g., `103447 / 173 U.S. 555`), neither pipeline can be verified against grep — both extracted some normalized form of a citation written differently in the source.

**Conclusion**: section *boundaries* are identical between the two pipelines. The disagreement comes from (a) eyecite's resolver finding more occurrences via back-reference chaining (the dominant pattern), and (b) Haiku's LLM mis-assigning sections near boundaries (the small remainder). There is a minor real bug in eyecite-pipeline: citations whose offset lands in the inter-section paragraph-break gap return `None` — but this is rare and not the primary cause of the disagreement you observed.

## Headline findings to date

1. **Haiku silently dropped 5 of 383 opinions** — all large SCOTUS opinions (245–382 KB). Eyecite handled them fine. Suggests Haiku's batch / pagination path needs investigation independent of the extractor choice.
2. **Eyecite dropped 1 opinion** — `4450553`, which is heavily NBSP-encoded. A real-world eyecite gap, but an order of magnitude smaller in scope than the Haiku gap.
3. **Eyecite extracts ~20% more total citations** (15,604 vs 13,044). Likely a mix of (a) eyecite over-extracts where the resolver fails to merge short/Id/supra back to a Full, and (b) Haiku is more conservative.
4. **Per-opinion overlap (Jaccard) is the key quality signal** — see distribution and lowest-agreement examples above.

Next analyses worth running: caseName-quality audit (eyecite truncates plaintiffs), ground-truth comparison against expert labels, Sonnet extraction added to the comparison.